<a href="https://colab.research.google.com/github/mgkagori/dissertation-ecommerce-edt/blob/Data-Samples/Sample_2_MCAuley_Data_set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd, json, urllib.request
from collections import defaultdict

BASE = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw"
META_URL   = f"{BASE}/meta_categories/meta_Electronics.jsonl"
REVIEW_URL = f"{BASE}/review_categories/Electronics.jsonl"

def stream_jsonl(url):
    """Yield JSON objects line-by-line from a remote .jsonl file."""
    with urllib.request.urlopen(url) as resp:
        for line in resp:
            yield json.loads(line)

In [2]:
first = next(stream_jsonl(META_URL))
print(first.get("parent_asin"), "|", first.get("title"))

B00MCW7G9M | FS-1051 FATSHARK TELEPORTER V3 HEADSET


In [3]:
products = {}
for m in stream_jsonl(META_URL):
    desc = " ".join(m.get("description") or [])
    if len(desc) >= 100:
        products[m["parent_asin"]] = {
            "parent_asin": m["parent_asin"],
            "product_title": m.get("title"),
            "description": desc,
            "price": m.get("price"),
            "store": m.get("store"),
        }
    if len(products) >= 5000:
        break

print(f"Collected {len(products)} described products")

Collected 5000 described products


In [4]:
target = set(products)
reviews_by_product = defaultdict(list)

for r in stream_jsonl(REVIEW_URL):
    pid = r.get("parent_asin")
    if pid in target:
        reviews_by_product[pid].append({
            "parent_asin": pid,
            "rating": r.get("rating"),
            "text": r.get("text"),
            "timestamp": r.get("timestamp"),
            "verified_purchase": r.get("verified_purchase"),
        })
    if sum(len(v) >= 20 for v in reviews_by_product.values()) >= 100:
        break

print(f"Products with >=20 reviews: "
      f"{sum(len(v) >= 20 for v in reviews_by_product.values())}")

Products with >=20 reviews: 100


In [5]:
MIN_REVIEWS = 20

rows = [{**rev, **products[pid]}
        for pid, revs in reviews_by_product.items() if len(revs) >= MIN_REVIEWS
        for rev in revs]

df = pd.DataFrame(rows)
print(f"{len(df)} reviews across {df['parent_asin'].nunique()} products")
df.head()

6184 reviews across 100 products


,parent_asin,rating,text,timestamp,verified_purchase,product_title,description,price,store
0,B0862W5BQ8,4.0,I believe the electrical boxes are made especi...,1484481615000,False,DATA COMM Electronics 45-6001-WH-S 1-Gang Rece...,DataComm Electronics’ 1-Gang Recessed Low Volt...,6.6,DATA COMM
1,B0862W5BQ8,4.0,"Nice item, will likely order others",1464828558000,True,DATA COMM Electronics 45-6001-WH-S 1-Gang Rece...,DataComm Electronics’ 1-Gang Recessed Low Volt...,6.6,DATA COMM
2,B0862W5BQ8,5.0,Hard to say much about a plastic faceplate oth...,1327163504000,True,DATA COMM Electronics 45-6001-WH-S 1-Gang Rece...,DataComm Electronics’ 1-Gang Recessed Low Volt...,6.6,DATA COMM
3,B0862W5BQ8,5.0,Perfect for running wires through your wall.,1469627308000,False,DATA COMM Electronics 45-6001-WH-S 1-Gang Rece...,DataComm Electronics’ 1-Gang Recessed Low Volt...,6.6,DATA COMM
4,B0862W5BQ8,5.0,"Installation was easy. For the DIY, it's no p...",1263096771000,True,DATA COMM Electronics 45-6001-WH-S 1-Gang Rece...,DataComm Electronics’ 1-Gang Recessed Low Volt...,6.6,DATA COMM


In [6]:
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/dissertation/final_electronics_merged_clean.parquet'
df.to_parquet(path)
df.head(50).to_csv(path.replace('.parquet', '_preview.csv'), index=False)
print(f"Saved {len(df)} reviews")

Mounted at /content/drive
Saved 6184 reviews


In [7]:
print(f"{len(df)} reviews across {df['parent_asin'].nunique()} products")

6184 reviews across 100 products


In [8]:
print(df.groupby('parent_asin').size().describe())
print(f"Products with >=20 reviews: {(df.groupby('parent_asin').size() >= 20).sum()}")

count    100.000000
mean      61.840000
std       56.058721
min       20.000000
25%       26.750000
50%       41.000000
75%       84.250000
max      327.000000
dtype: float64
Products with >=20 reviews: 100
